In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
VISUALISASI HASIL UNDUHAN 25.000 EVENT
- Peta sebaran event berhasil vs gagal
- Distribusi stasiun
- Timeline per tahun
- Success rate per tahun
"""

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# =============================================
# 1. KONFIGURASI
# =============================================

WAVEFORM_DIR = "/Volumes/Extreme SSD/unduhan_waveform_geofon_juli"
CATALOG_CSV = "/Volumes/Extreme SSD/unduhan_juni_bmkg_usgs/hasilscan/eda_final_output/HYBRID_EARTHQUAKE_CATALOG_2001_2024_FIX.csv"
OUTPUT_DIR = "visualisasi_unduhan"
Path(OUTPUT_DIR).mkdir(exist_ok=True, parents=True)

# =============================================
# 2. BACA DATA
# =============================================

print("="*70)
print("📊 VISUALISASI HASIL UNDUHAN 25.000 EVENT")
print("="*70)

# --- 2a. Baca file .mseed ---
print("\n📂 Membaca file .mseed...")
files = list(Path(WAVEFORM_DIR).glob("*.mseed"))
print(f"✅ Total file terunduh: {len(files)}")

# Ekstrak metadata dari nama file
file_data = []
for f in files:
    parts = f.stem.split('_')
    if len(parts) >= 3:
        network = parts[0]
        station = parts[1]
        # event_id adalah sisa setelah network_station
        event_id = '_'.join(parts[2:]) if len(parts) > 2 else 'unknown'
        # Ambil tanggal dari event_id (format: YYYYMMDD_HHMMSS)
        try:
            date_str = event_id[:8] if len(event_id) >= 8 else '00000000'
            year = int(date_str[:4]) if date_str[:4].isdigit() else 0
        except:
            year = 0
        file_data.append({
            'file': f.name,
            'network': network,
            'station': station,
            'event_id': event_id,
            'year': year,
            'size_kb': f.stat().st_size / 1024
        })

df_files = pd.DataFrame(file_data)
print(f"✅ Metadata diekstrak dari {len(df_files)} file")

# --- 2b. Baca katalog untuk informasi event ---
print("\n📂 Membaca katalog...")
if os.path.exists(CATALOG_CSV):
    df_catalog = pd.read_csv(CATALOG_CSV)
    df_catalog['datetime'] = pd.to_datetime(df_catalog['time_utc'], utc=True)
    df_catalog['year'] = df_catalog['datetime'].dt.year
    df_catalog['event_id'] = df_catalog['event_id'].astype(str)
    print(f"✅ Katalog dimuat: {len(df_catalog)} event")
    
    # Merge dengan file data berdasarkan event_id
    # Karena format event_id berbeda, kita coba match dengan substring
    # Event_id di file: usp000a716, di katalog: usp000a716
    # Sebagian file mungkin punya event_id dengan format lain
    df_files['event_id_clean'] = df_files['event_id'].str.split('_').str[0]
    df_merged = df_files.merge(df_catalog, left_on='event_id_clean', right_on='event_id', how='left')
    print(f"✅ Merge dengan katalog: {len(df_merged)} event")
else:
    print("⚠️ Katalog tidak ditemukan, hanya akan menampilkan distribusi stasiun")
    df_merged = df_files.copy()
    df_merged['latitude'] = np.nan
    df_merged['longitude'] = np.nan
    df_merged['magnitude'] = np.nan
    df_merged['depth_km'] = np.nan

# =============================================
# 3. STATISTIK DASAR
# =============================================

print("\n" + "="*70)
print("📊 STATISTIK DASAR")
print("="*70)

total_files = len(df_files)
total_events = 25000
success_rate = total_files / total_events * 100

print(f"Total event diproses: {total_events}")
print(f"Total file terunduh: {total_files}")
print(f"Success rate: {success_rate:.1f}%")
print(f"Gagal: {total_events - total_files} ({100-success_rate:.1f}%)")

# Distribusi jaringan
print("\n📡 10 Jaringan Terbanyak:")
for net, count in df_files['network'].value_counts().head(10).items():
    print(f"  {net}: {count} ({count/total_files*100:.1f}%)")

# Distribusi stasiun
print("\n📡 15 Stasiun Terbanyak:")
for sta, count in df_files['station'].value_counts().head(15).items():
    print(f"  {sta}: {count} ({count/total_files*100:.1f}%)")

# =============================================
# 4. VISUALISASI
# =============================================

sns.set_style("whitegrid")
sns.set_palette("viridis")
plt.rcParams['font.size'] = 10

# --- 4a. Distribusi Stasiun (Top 15) ---
fig, ax = plt.subplots(figsize=(14, 6))
top_stations = df_files['station'].value_counts().head(15)
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(top_stations)))
bars = ax.bar(top_stations.index, top_stations.values, color=colors, alpha=0.8)
ax.set_xlabel('Stasiun', fontsize=12)
ax.set_ylabel('Jumlah File', fontsize=12)
ax.set_title('15 Stasiun dengan File Terbanyak', fontsize=14)
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, top_stations.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 10, f'{val}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/top_stations.png", dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ Top stations: {OUTPUT_DIR}/top_stations.png")

# --- 4b. Pie Chart Jaringan ---
fig, ax = plt.subplots(figsize=(8, 8))
network_counts = df_files['network'].value_counts()
# Gabungkan yang kecil ke 'Other'
threshold = 2.0
others = network_counts[network_counts / total_files * 100 < threshold].sum()
main_networks = network_counts[network_counts / total_files * 100 >= threshold]
if others > 0:
    main_networks['Other'] = others

colors_pie = plt.cm.tab10(np.linspace(0, 1, len(main_networks)))
wedges, texts, autotexts = ax.pie(main_networks.values, labels=main_networks.index,
                                   autopct='%1.1f%%', colors=colors_pie,
                                   startangle=90, explode=[0.02]*len(main_networks))
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')
ax.set_title('Distribusi Jaringan Stasiun', fontsize=14)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/network_distribution.png", dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ Network distribution: {OUTPUT_DIR}/network_distribution.png")

# --- 4c. Timeline per Tahun (jika ada data tahun) ---
if 'year' in df_files.columns and df_files['year'].sum() > 0:
    fig, ax = plt.subplots(figsize=(14, 6))
    yearly = df_files['year'].value_counts().sort_index()
    bars = ax.bar(yearly.index, yearly.values, color='steelblue', alpha=0.7)
    ax.set_xlabel('Tahun', fontsize=12)
    ax.set_ylabel('Jumlah File Terunduh', fontsize=12)
    ax.set_title('Distribusi File Terunduh per Tahun', fontsize=14)
    ax.grid(True, alpha=0.3, axis='y')
    # Tambahkan nilai di atas bar
    for bar, val in zip(bars, yearly.values):
        ax.text(bar.get_x() + bar.get_width()/2, val + 20, f'{val}', ha='center', va='bottom', fontsize=8)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/timeline_yearly.png", dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Timeline tahunan: {OUTPUT_DIR}/timeline_yearly.png")

# --- 4d. Peta Sebaran (jika ada koordinat) ---
if 'latitude' in df_merged.columns and df_merged['latitude'].notna().sum() > 0:
    try:
        import cartopy.crs as ccrs
        import cartopy.feature as cfeature
        
        fig, ax = plt.subplots(figsize=(14, 10), subplot_kw={'projection': ccrs.PlateCarree()})
        ax.set_extent([90, 145, -12, 8], crs=ccrs.PlateCarree())
        
        # Tambahkan fitur peta
        ax.add_feature(cfeature.LAND, facecolor='lightgray', edgecolor='black', linewidth=0.5)
        ax.add_feature(cfeature.OCEAN, facecolor='white', alpha=0.5)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
        ax.add_feature(cfeature.BORDERS, linewidth=0.5, alpha=0.5)
        ax.add_feature(cfeature.LAKES, facecolor='lightblue', edgecolor='none')
        
        # Plot event yang berhasil diunduh
        df_merged_valid = df_merged[df_merged['latitude'].notna() & df_merged['longitude'].notna()]
        if len(df_merged_valid) > 0:
            scatter = ax.scatter(df_merged_valid['longitude'], df_merged_valid['latitude'],
                                 c=df_merged_valid.get('magnitude', np.ones(len(df_merged_valid))*5),
                                 s=3, cmap='viridis', alpha=0.5, transform=ccrs.PlateCarree())
            cbar = plt.colorbar(scatter, ax=ax, orientation='vertical', shrink=0.6, pad=0.05)
            cbar.set_label('Magnitudo')
        
        ax.set_title(f'Peta Sebaran Event Terunduh ({len(df_merged_valid)} event)', fontsize=14)
        ax.gridlines(draw_labels=True, linestyle='--', alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f"{OUTPUT_DIR}/spatial_map.png", dpi=300, bbox_inches='tight')
        plt.close()
        print(f"🗺️  Peta sebaran: {OUTPUT_DIR}/spatial_map.png")
    except ImportError:
        print("⚠️ Cartopy tidak tersedia, lewati pembuatan peta")

# --- 4e. Ukuran File ---
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(df_files['size_kb'], bins=50, color='steelblue', alpha=0.7, edgecolor='black')
ax.axvline(df_files['size_kb'].median(), color='red', linestyle='--', linewidth=2, label=f'Median: {df_files["size_kb"].median():.1f} KB')
ax.set_xlabel('Ukuran File (KB)', fontsize=12)
ax.set_ylabel('Frekuensi', fontsize=12)
ax.set_title('Distribusi Ukuran File .mseed', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/file_size_distribution.png", dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ Ukuran file: {OUTPUT_DIR}/file_size_distribution.png")

# --- 4f. Summary Table ---
fig, ax = plt.subplots(figsize=(12, 4))
ax.axis('tight')
ax.axis('off')

summary_data = [
    ['Total event diproses', str(total_events)],
    ['Berhasil diunduh', f"{total_files} ({success_rate:.1f}%)"],
    ['Gagal', f"{total_events - total_files} ({100-success_rate:.1f}%)"],
    ['Jaringan terbanyak', df_files['network'].value_counts().index[0]],
    ['Stasiun terbanyak', df_files['station'].value_counts().index[0]],
    ['Waktu unduhan', '10 jam 41 menit'],
    ['Rata-rata per event', '1.54 detik'],
]

table = ax.table(cellText=summary_data, colLabels=['Metrik', 'Nilai'],
                 cellLoc='center', loc='center',
                 colColours=['#4472C4', '#4472C4'])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 1.8)
ax.set_title('Ringkasan Unduhan 25.000 Event', fontsize=14, pad=20)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/summary_table.png", dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ Summary table: {OUTPUT_DIR}/summary_table.png")

# =============================================
# 5. RINGKASAN
# =============================================

print("\n" + "="*70)
print("📊 RINGKASAN VISUALISASI")
print("="*70)
print(f"Total file terunduh: {total_files}")
print(f"Success rate: {success_rate:.1f}%")
print(f"Jaringan terbanyak: {df_files['network'].value_counts().index[0]} ({df_files['network'].value_counts().iloc[0]} file)")
print(f"Stasiun terbanyak: {df_files['station'].value_counts().index[0]} ({df_files['station'].value_counts().iloc[0]} file)")
if 'year' in df_files.columns and df_files['year'].sum() > 0:
    year_min = df_files['year'][df_files['year'] > 0].min()
    year_max = df_files['year'].max()
    print(f"Rentang tahun data: {year_min} - {year_max}")
print(f"Total storage: {df_files['size_kb'].sum() / (1024*1024):.2f} GB")
print(f"Rata-rata ukuran file: {df_files['size_kb'].mean():.1f} KB")
print("="*70)
print(f"\n✅ Semua visualisasi disimpan di folder: {OUTPUT_DIR}/")

📊 VISUALISASI HASIL UNDUHAN 25.000 EVENT

📂 Membaca file .mseed...
✅ Total file terunduh: 40430
✅ Metadata diekstrak dari 40430 file

📂 Membaca katalog...
✅ Katalog dimuat: 232301 event
✅ Merge dengan katalog: 40430 event

📊 STATISTIK DASAR
Total event diproses: 25000
Total file terunduh: 40430
Success rate: 161.7%
Gagal: -15430 (-61.7%)

📡 10 Jaringan Terbanyak:
  .: 20215 (50.0%)
  GE: 19101 (47.2%)
  XN: 641 (1.6%)
  Z6: 444 (1.1%)
  7A: 29 (0.1%)

📡 15 Stasiun Terbanyak:
  GE: 19101 (47.2%)
  UGM: 5624 (13.9%)
  PMG: 2586 (6.4%)
  TNTI: 2420 (6.0%)
  GSI: 1709 (4.2%)
  MNAI: 1044 (2.6%)
  BNDI: 892 (2.2%)
  MMRI: 839 (2.1%)
  LUWI: 686 (1.7%)
  XN: 641 (1.6%)
  FAKI: 636 (1.6%)
  CISI: 535 (1.3%)
  Z6: 444 (1.1%)
  YOGI: 390 (1.0%)
  BUM: 384 (0.9%)
✅ Top stations: visualisasi_unduhan/top_stations.png
✅ Network distribution: visualisasi_unduhan/network_distribution.png
✅ Timeline tahunan: visualisasi_unduhan/timeline_yearly.png
✅ Ukuran file: visualisasi_unduhan/file_size_distribut

In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
VISUALISASI HASIL UNDUHAN - FIXED
- Filter file ._ (metadata macOS)
- Filter file dengan ukuran < 1 KB (corrupt)
- Deteksi duplikat event
- Hitung success rate yang benar
"""

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# =============================================
# 1. KONFIGURASI
# =============================================

WAVEFORM_DIR = "/Volumes/Extreme SSD/unduhan_waveform_geofon_juli"
CATALOG_CSV = "/Volumes/Extreme SSD/unduhan_juni_bmkg_usgs/hasilscan/eda_final_output/HYBRID_EARTHQUAKE_CATALOG_2001_2024_FIX.csv"
OUTPUT_DIR = "visualisasi_unduhan_fixed"
Path(OUTPUT_DIR).mkdir(exist_ok=True, parents=True)

# =============================================
# 2. BACA & BERSIHKAN DATA
# =============================================

print("="*70)
print("📊 VISUALISASI HASIL UNDUHAN (FIXED)")
print("="*70)

# --- 2a. Baca file .mseed dengan filter ---
print("\n📂 Membaca file .mseed...")
all_files = list(Path(WAVEFORM_DIR).glob("*.mseed"))
print(f"Total file ditemukan: {len(all_files)}")

# Filter: exclude file ._ (macOS metadata)
valid_files = [f for f in all_files if not f.name.startswith('._')]
print(f"File setelah exclude ._: {len(valid_files)}")

# Filter: exclude file dengan ukuran < 1 KB (corrupt)
valid_files = [f for f in valid_files if f.stat().st_size > 1024]
print(f"File setelah filter ukuran > 1 KB: {len(valid_files)}")

# Filter: exclude file yang tidak sesuai format nama (minimal 3 bagian)
valid_files = [f for f in valid_files if len(f.stem.split('_')) >= 3]
print(f"File setelah filter format nama: {len(valid_files)}")

# Ekstrak metadata
file_data = []
for f in valid_files:
    parts = f.stem.split('_')
    if len(parts) >= 3:
        # Format: NETWORK_STATION_EVENTID
        # Tapi untuk file dari GE, formatnya: GE_STATION_eventid
        network = parts[0]
        station = parts[1]
        event_id = '_'.join(parts[2:])
        
        # Coba ambil tahun dari event_id
        try:
            # Coba pattern YYYYMMDD di awal event_id
            import re
            match = re.search(r'(\d{4})(\d{2})(\d{2})', event_id)
            if match:
                year = int(match.group(1))
            else:
                year = 0
        except:
            year = 0
        
        file_data.append({
            'file': f.name,
            'network': network,
            'station': station,
            'event_id': event_id,
            'year': year,
            'size_kb': f.stat().st_size / 1024,
            'size_mb': f.stat().st_size / (1024*1024),
            'path': str(f)
        })

df_files = pd.DataFrame(file_data)
print(f"✅ Metadata diekstrak dari {len(df_files)} file")

# --- 2b. Deteksi duplikat event (multiple files per event) ---
event_counts = df_files['event_id'].value_counts()
unique_events = len(event_counts)
duplicates = event_counts[event_counts > 1].sum()
print(f"\n✅ Event unik: {unique_events}")
print(f"   File duplikat: {duplicates} (file tambahan untuk event yang sama)")

# Ambil satu file per event (yang terbaik = ukuran terbesar)
df_best = df_files.sort_values(['event_id', 'size_kb'], ascending=[True, False])
df_unique = df_best.drop_duplicates(subset=['event_id'], keep='first')
print(f"✅ Event unik setelah deduplikasi: {len(df_unique)}")

# =============================================
# 3. LOAD KATALOG UNTUK MERGE
# =============================================

print("\n📂 Membaca katalog...")
if os.path.exists(CATALOG_CSV):
    df_catalog = pd.read_csv(CATALOG_CSV)
    df_catalog['datetime'] = pd.to_datetime(df_catalog['time_utc'], utc=True)
    df_catalog['year'] = df_catalog['datetime'].dt.year
    df_catalog['event_id'] = df_catalog['event_id'].astype(str)
    print(f"✅ Katalog dimuat: {len(df_catalog)} event")
    
    # Merge dengan event_id
    df_merged = df_unique.merge(df_catalog, left_on='event_id', right_on='event_id', how='left')
    print(f"✅ Merge dengan katalog: {len(df_merged)} event")
else:
    print("⚠️ Katalog tidak ditemukan")
    df_merged = df_unique.copy()
    df_merged['latitude'] = np.nan
    df_merged['longitude'] = np.nan
    df_merged['magnitude'] = np.nan
    df_merged['depth_km'] = np.nan

# =============================================
# 4. STATISTIK YANG BENAR
# =============================================

print("\n" + "="*70)
print("📊 STATISTIK YANG BENAR")
print("="*70)

total_events_target = 25000
unique_events_downloaded = len(df_unique)
success_rate = unique_events_downloaded / total_events_target * 100

print(f"Total event target: {total_events_target}")
print(f"Event unik berhasil diunduh: {unique_events_downloaded}")
print(f"Success rate: {success_rate:.1f}%")
print(f"Event gagal (tidak ada file): {total_events_target - unique_events_downloaded} ({100-success_rate:.1f}%)")
print(f"Total file (termasuk duplikat): {len(df_files)}")
print(f"Rata-rata file per event: {len(df_files) / unique_events_downloaded:.2f}")

# Distribusi stasiun (hanya file valid)
station_counts = df_unique['station'].value_counts()
network_counts = df_unique['network'].value_counts()

print("\n📡 10 Stasiun Terbanyak:")
for sta, count in station_counts.head(10).items():
    print(f"  {sta}: {count} ({count/unique_events_downloaded*100:.1f}%)")

print("\n📡 5 Jaringan Terbanyak:")
for net, count in network_counts.head(5).items():
    print(f"  {net}: {count} ({count/unique_events_downloaded*100:.1f}%)")

# Ukuran file
print(f"\n💾 Total storage (unik): {df_unique['size_mb'].sum():.1f} MB ({df_unique['size_mb'].sum()/1024:.2f} GB)")
print(f"Rata-rata ukuran file: {df_unique['size_kb'].mean():.1f} KB")

# =============================================
# 5. VISUALISASI
# =============================================

sns.set_style("whitegrid")
plt.rcParams['font.size'] = 10

# --- 5a. Top Stations ---
fig, ax = plt.subplots(figsize=(14, 6))
top_stations = station_counts.head(15)
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(top_stations)))
bars = ax.bar(top_stations.index, top_stations.values, color=colors, alpha=0.8)
ax.set_xlabel('Stasiun', fontsize=12)
ax.set_ylabel('Jumlah Event', fontsize=12)
ax.set_title(f'15 Stasiun dengan Event Terbanyak (dari {unique_events_downloaded} event)', fontsize=14)
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, top_stations.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 5, f'{val}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/top_stations.png", dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ Top stations: {OUTPUT_DIR}/top_stations.png")

# --- 5b. Network Pie ---
fig, ax = plt.subplots(figsize=(8, 8))
network_counts = df_unique['network'].value_counts()
threshold = 2.0
others = network_counts[network_counts / unique_events_downloaded * 100 < threshold].sum()
main_networks = network_counts[network_counts / unique_events_downloaded * 100 >= threshold]
if others > 0:
    main_networks['Other'] = others

colors_pie = plt.cm.tab10(np.linspace(0, 1, len(main_networks)))
wedges, texts, autotexts = ax.pie(main_networks.values, labels=main_networks.index,
                                   autopct='%1.1f%%', colors=colors_pie,
                                   startangle=90, explode=[0.02]*len(main_networks))
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')
ax.set_title('Distribusi Jaringan Stasiun', fontsize=14)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/network_distribution.png", dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ Network distribution: {OUTPUT_DIR}/network_distribution.png")

# --- 5c. Timeline ---
if 'year' in df_unique.columns and df_unique['year'].sum() > 0:
    fig, ax = plt.subplots(figsize=(14, 6))
    # Filter tahun valid (> 1900)
    df_years = df_unique[df_unique['year'] > 1900]
    yearly = df_years['year'].value_counts().sort_index()
    if len(yearly) > 0:
        bars = ax.bar(yearly.index, yearly.values, color='steelblue', alpha=0.7)
        ax.set_xlabel('Tahun', fontsize=12)
        ax.set_ylabel('Jumlah Event', fontsize=12)
        ax.set_title('Distribusi Event per Tahun (Unik)', fontsize=14)
        ax.grid(True, alpha=0.3, axis='y')
        for bar, val in zip(bars, yearly.values):
            ax.text(bar.get_x() + bar.get_width()/2, val + 5, f'{val}', ha='center', va='bottom', fontsize=8)
        plt.tight_layout()
        plt.savefig(f"{OUTPUT_DIR}/timeline_yearly.png", dpi=300, bbox_inches='tight')
        plt.close()
        print(f"✅ Timeline tahunan: {OUTPUT_DIR}/timeline_yearly.png")

# --- 5d. Ukuran File ---
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(df_unique['size_kb'], bins=50, color='steelblue', alpha=0.7, edgecolor='black')
ax.axvline(df_unique['size_kb'].median(), color='red', linestyle='--', linewidth=2, 
           label=f'Median: {df_unique["size_kb"].median():.1f} KB')
ax.set_xlabel('Ukuran File (KB)', fontsize=12)
ax.set_ylabel('Frekuensi', fontsize=12)
ax.set_title('Distribusi Ukuran File (Event Unik)', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/file_size_distribution.png", dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ Ukuran file: {OUTPUT_DIR}/file_size_distribution.png")

# --- 5e. Summary Table ---
fig, ax = plt.subplots(figsize=(12, 4))
ax.axis('tight')
ax.axis('off')

total_files_all = len(all_files)
files_after_filter = len(valid_files)
unique_events_count = len(df_unique)

summary_data = [
    ['Total event target', str(total_events_target)],
    ['Event unik berhasil', f"{unique_events_count} ({success_rate:.1f}%)"],
    ['Total file (termasuk duplikat)', str(len(df_files))],
    ['File corrupt / invalid', str(len(all_files) - len(valid_files))],
    ['Stasiun terbanyak', station_counts.index[0] if len(station_counts) > 0 else 'N/A'],
    ['Jaringan terbanyak', network_counts.index[0] if len(network_counts) > 0 else 'N/A'],
    ['Total storage', f"{df_unique['size_mb'].sum()/1024:.2f} GB"],
]

table = ax.table(cellText=summary_data, colLabels=['Metrik', 'Nilai'],
                 cellLoc='center', loc='center',
                 colColours=['#4472C4', '#4472C4'])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 1.8)
ax.set_title('Ringkasan Unduhan 25.000 Event (Setelah Pembersihan)', fontsize=14, pad=20)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/summary_table.png", dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ Summary table: {OUTPUT_DIR}/summary_table.png")

# =============================================
# 6. RINGKASAN AKHIR
# =============================================

print("\n" + "="*70)
print("📊 RINGKASAN AKHIR")
print("="*70)
print(f"Event target: {total_events_target}")
print(f"Event unik berhasil diunduh: {unique_events_count} ({success_rate:.1f}%)")
print(f"Event gagal: {total_events_target - unique_events_count} ({100-success_rate:.1f}%)")
print(f"Total file (termasuk duplikat): {len(df_files)}")
print(f"Rata-rata file per event: {len(df_files) / unique_events_count:.2f}")
print(f"Total storage: {df_unique['size_mb'].sum()/1024:.2f} GB")
print(f"Rata-rata ukuran file: {df_unique['size_kb'].mean():.1f} KB")
print("="*70)

# Simpan daftar event unik
df_unique[['event_id', 'network', 'station', 'year', 'size_mb']].to_csv(f"{OUTPUT_DIR}/unique_events.csv", index=False)
print(f"\n✅ Daftar event unik tersimpan: {OUTPUT_DIR}/unique_events.csv")
print(f"\n✅ Semua visualisasi disimpan di folder: {OUTPUT_DIR}/")

📊 VISUALISASI HASIL UNDUHAN (FIXED)

📂 Membaca file .mseed...
Total file ditemukan: 40430
File setelah exclude ._: 20215
File setelah filter ukuran > 1 KB: 20212
File setelah filter format nama: 20212
✅ Metadata diekstrak dari 20212 file

✅ Event unik: 20211
   File duplikat: 2 (file tambahan untuk event yang sama)
✅ Event unik setelah deduplikasi: 20211

📂 Membaca katalog...
✅ Katalog dimuat: 232301 event
✅ Merge dengan katalog: 20211 event

📊 STATISTIK YANG BENAR
Total event target: 25000
Event unik berhasil diunduh: 20211
Success rate: 80.8%
Event gagal (tidak ada file): 4789 (19.2%)
Total file (termasuk duplikat): 20212
Rata-rata file per event: 1.00

📡 10 Stasiun Terbanyak:
  UGM: 5624 (27.8%)
  PMG: 2586 (12.8%)
  TNTI: 2419 (12.0%)
  GSI: 1709 (8.5%)
  MNAI: 1044 (5.2%)
  BNDI: 892 (4.4%)
  MMRI: 836 (4.1%)
  LUWI: 686 (3.4%)
  FAKI: 636 (3.1%)
  CISI: 535 (2.6%)

📡 5 Jaringan Terbanyak:
  GE: 19097 (94.5%)
  XN: 641 (3.2%)
  Z6: 444 (2.2%)
  7A: 29 (0.1%)

💾 Total storage (unik

In [11]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
ANALISIS FILE .mseed YANG BERHASIL DIUNDUH
Tanpa mencocokkan dengan katalog (karena format berbeda)
"""

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# =============================================
# 1. KONFIGURASI
# =============================================

WAVEFORM_DIR = "/Volumes/Extreme SSD/unduhan_waveform_geofon_juli"
OUTPUT_DIR = "analisis_file_terunduh"
Path(OUTPUT_DIR).mkdir(exist_ok=True, parents=True)

# =============================================
# 2. BACA FILE .mseed
# =============================================

print("="*70)
print("📊 ANALISIS FILE .mseed YANG BERHASIL DIUNDUH")
print("="*70)

print("\n📂 Membaca file .mseed...")
files = list(Path(WAVEFORM_DIR).glob("*.mseed"))
files = [f for f in files if not f.name.startswith('._')]
print(f"✅ Total file .mseed valid: {len(files)}")

# =============================================
# 3. EKSTRAK METADATA DARI NAMA FILE
# =============================================

print("\n📊 Mengekstrak metadata dari nama file...")

file_data = []
for f in files:
    parts = f.stem.split('_')
    if len(parts) >= 3:
        network = parts[0]
        station = parts[1]
        # Gabungkan sisa sebagai event_id
        event_id = '_'.join(parts[2:]) if len(parts) > 2 else 'unknown'
        
        # Ekstrak tahun dari event_id (format: YYYYMMDD atau YYYYMMDDHHMMSS)
        year = None
        month = None
        day = None
        hour = None
        minute = None
        second = None
        
        # Coba ekstrak dari event_id
        for p in parts[2:]:
            if len(p) >= 8 and p[:8].isdigit():
                year = int(p[:4])
                month = int(p[4:6])
                day = int(p[6:8])
                if len(p) >= 14:
                    hour = int(p[8:10])
                    minute = int(p[10:12])
                    second = int(p[12:14])
                break
        
        file_data.append({
            'file': f.name,
            'network': network,
            'station': station,
            'event_id': event_id,
            'size_kb': f.stat().st_size / 1024,
            'year': year,
            'month': month,
            'day': day,
            'hour': hour,
            'minute': minute,
            'second': second
        })

df = pd.DataFrame(file_data)
print(f"✅ Metadata diekstrak dari {len(df)} file")

# =============================================
# 4. STATISTIK DASAR
# =============================================

print("\n" + "="*70)
print("📊 STATISTIK DASAR")
print("="*70)

total_files = len(df)
print(f"Total file: {total_files}")

# --- Jaringan ---
print("\n📡 10 Jaringan Terbanyak:")
for net, count in df['network'].value_counts().head(10).items():
    print(f"  {net}: {count} ({count/total_files*100:.1f}%)")

# --- Stasiun ---
print("\n📡 15 Stasiun Terbanyak:")
for sta, count in df['station'].value_counts().head(15).items():
    print(f"  {sta}: {count} ({count/total_files*100:.1f}%)")

# --- Ukuran file ---
print(f"\n💾 Ukuran File:")
print(f"  Total storage: {df['size_kb'].sum() / (1024*1024):.2f} GB")
print(f"  Rata-rata: {df['size_kb'].mean():.1f} KB")
print(f"  Median: {df['size_kb'].median():.1f} KB")
print(f"  Min: {df['size_kb'].min():.1f} KB")
print(f"  Max: {df['size_kb'].max():.1f} KB")

# =============================================
# 5. DISTRIBUSI TAHUN (jika ada)
# =============================================

df_valid_year = df[df['year'].notna()]
if len(df_valid_year) > 0:
    print(f"\n📊 Distribusi Tahun (dari {len(df_valid_year)} file):")
    yearly = df_valid_year['year'].value_counts().sort_index()
    for year, count in yearly.items():
        print(f"  {year}: {count}")

# =============================================
# 6. VISUALISASI
# =============================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 6a. Top stations
ax1 = axes[0, 0]
top_stations = df['station'].value_counts().head(15)
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(top_stations)))
bars = ax1.barh(top_stations.index, top_stations.values, color=colors, alpha=0.8)
ax1.set_xlabel('Jumlah File')
ax1.set_title('15 Stasiun dengan File Terbanyak')
ax1.grid(True, alpha=0.3, axis='x')
for bar, val in zip(bars, top_stations.values):
    ax1.text(val + 10, bar.get_y() + bar.get_height()/2, f'{val}', va='center', fontsize=8)

# 6b. Top networks (pie)
ax2 = axes[0, 1]
network_counts = df['network'].value_counts()
# Gabungkan yang kecil ke 'Other'
threshold = 2.0
others = network_counts[network_counts / total_files * 100 < threshold].sum()
main_networks = network_counts[network_counts / total_files * 100 >= threshold]
if others > 0:
    main_networks['Other'] = others

colors_pie = plt.cm.tab10(np.linspace(0, 1, len(main_networks)))
wedges, texts, autotexts = ax2.pie(main_networks.values, labels=main_networks.index,
                                   autopct='%1.1f%%', colors=colors_pie,
                                   startangle=90, explode=[0.02]*len(main_networks))
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')
ax2.set_title('Distribusi Jaringan', fontsize=12)

# 6c. File size distribution
ax3 = axes[1, 0]
ax3.hist(df['size_kb'], bins=50, color='steelblue', alpha=0.7, edgecolor='black')
ax3.axvline(df['size_kb'].median(), color='red', linestyle='--', linewidth=2,
            label=f'Median: {df["size_kb"].median():.1f} KB')
ax3.set_xlabel('Ukuran File (KB)')
ax3.set_ylabel('Frekuensi')
ax3.set_title('Distribusi Ukuran File')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 6d. Timeline tahunan (jika ada)
ax4 = axes[1, 1]
if len(df_valid_year) > 0:
    yearly = df_valid_year['year'].value_counts().sort_index()
    ax4.bar(yearly.index, yearly.values, color='coral', alpha=0.7, edgecolor='black')
    ax4.set_xlabel('Tahun')
    ax4.set_ylabel('Jumlah File')
    ax4.set_title('Distribusi File per Tahun')
    ax4.grid(True, alpha=0.3)
    # Tambahkan nilai
    for bar, val in zip(ax4.patches, yearly.values):
        ax4.text(bar.get_x() + bar.get_width()/2, val + 5, f'{val}', ha='center', va='bottom', fontsize=8)
else:
    ax4.text(0.5, 0.5, 'Tidak ada data tahun', ha='center', va='center', transform=ax4.transAxes)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/analisis_file_terunduh.png", dpi=300, bbox_inches='tight')
plt.close()
print(f"📊 Grafik tersimpan: {OUTPUT_DIR}/analisis_file_terunduh.png")

# =============================================
# 7. SIMPAN DATA
# =============================================

df.to_csv(f"{OUTPUT_DIR}/file_metadata.csv", index=False)
print(f"✅ Metadata file tersimpan: {OUTPUT_DIR}/file_metadata.csv")

# =============================================
# 8. RINGKASAN
# =============================================

print("\n" + "="*70)
print("📊 RINGKASAN ANALISIS")
print("="*70)
print(f"Total file terunduh: {total_files}")
print(f"Total storage: {df['size_kb'].sum() / (1024*1024):.2f} GB")
print(f"Jaringan terbanyak: {df['network'].value_counts().index[0]} ({df['network'].value_counts().iloc[0]} file)")
print(f"Stasiun terbanyak: {df['station'].value_counts().index[0]} ({df['station'].value_counts().iloc[0]} file)")
if len(df_valid_year) > 0:
    year_min = df_valid_year['year'].min()
    year_max = df_valid_year['year'].max()
    print(f"Rentang tahun: {year_min} - {year_max}")
print("="*70)

print(f"\n✅ Semua hasil disimpan di: {OUTPUT_DIR}/")

📊 ANALISIS FILE .mseed YANG BERHASIL DIUNDUH

📂 Membaca file .mseed...
✅ Total file .mseed valid: 20215

📊 Mengekstrak metadata dari nama file...
✅ Metadata diekstrak dari 20215 file

📊 STATISTIK DASAR
Total file: 20215

📡 10 Jaringan Terbanyak:
  GE: 19101 (94.5%)
  XN: 641 (3.2%)
  Z6: 444 (2.2%)
  7A: 29 (0.1%)

📡 15 Stasiun Terbanyak:
  UGM: 5624 (27.8%)
  PMG: 2586 (12.8%)
  TNTI: 2420 (12.0%)
  GSI: 1709 (8.5%)
  MNAI: 1044 (5.2%)
  BNDI: 892 (4.4%)
  MMRI: 839 (4.2%)
  LUWI: 686 (3.4%)
  FAKI: 636 (3.1%)
  CISI: 535 (2.6%)
  YOGI: 390 (1.9%)
  BUM: 384 (1.9%)
  TOLI: 359 (1.8%)
  BKNI: 358 (1.8%)
  JAGI: 272 (1.3%)

💾 Ukuran File:
  Total storage: 0.87 GB
  Rata-rata: 45.0 KB
  Median: 24.0 KB
  Min: 1.0 KB
  Max: 211.0 KB

📊 Distribusi Tahun (dari 20215 file):
  2001: 1089
  2002: 1242
  2003: 670
  2004: 2020
  2005: 2712
  2006: 2479
  2007: 2060
  2008: 1764
  2009: 3858
  2010: 2321
📊 Grafik tersimpan: analisis_file_terunduh/analisis_file_terunduh.png
✅ Metadata file tersim

In [12]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
VERIFIKASI KECOCOKAN - COCOKKAN BERDASARKAN TANGGAL (1 HARI)
Karena format event_id berbeda, gunakan waktu untuk mencocokkan.
"""

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import timedelta
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# =============================================
# 1. KONFIGURASI
# =============================================

WAVEFORM_DIR = "/Volumes/Extreme SSD/unduhan_waveform_geofon_juli"
CATALOG_CSV = "/Volumes/Extreme SSD/unduhan_juni_bmkg_usgs/hasilscan/eda_final_output/HYBRID_EARTHQUAKE_CATALOG_2001_2024_FIX.csv"
OUTPUT_DIR = "verifikasi_katalog_v3"
Path(OUTPUT_DIR).mkdir(exist_ok=True, parents=True)

TIME_TOLERANCE_HOURS = 12  # toleransi 12 jam

# =============================================
# 2. BACA DATA
# =============================================

print("="*70)
print("🔍 VERIFIKASI KECOCOKAN (BERDASARKAN TANGGAL + 12 JAM)")
print("="*70)

# --- 2a. Baca file .mseed ---
print("\n📂 Membaca file .mseed...")
files = list(Path(WAVEFORM_DIR).glob("*.mseed"))
files = [f for f in files if not f.name.startswith('._')]
print(f"✅ Total file .mseed valid: {len(files)}")

# Ekstrak waktu dari nama file
downloaded_events = []
for f in files:
    parts = f.stem.split('_')
    if len(parts) >= 3:
        for p in parts:
            if len(p) == 14 and p.isdigit():
                try:
                    dt = pd.to_datetime(p, format='%Y%m%d%H%M%S', utc=True)
                    downloaded_events.append({
                        'file': f.name,
                        'network': parts[0],
                        'station': parts[1],
                        'datetime': dt,
                        'date': dt.date(),
                        'year': dt.year
                    })
                    break
                except:
                    continue
            elif len(p) == 8 and p.isdigit():
                try:
                    dt = pd.to_datetime(p, format='%Y%m%d', utc=True)
                    downloaded_events.append({
                        'file': f.name,
                        'network': parts[0],
                        'station': parts[1],
                        'datetime': dt,
                        'date': dt.date(),
                        'year': dt.year
                    })
                    break
                except:
                    continue

df_files = pd.DataFrame(downloaded_events)
print(f"✅ Event unik terunduh: {len(df_files)}")

if len(df_files) == 0:
    print("❌ Tidak ada event yang bisa diekstrak!")
    exit()

# --- 2b. Baca katalog ---
print("\n📂 Membaca katalog...")
df_catalog = pd.read_csv(CATALOG_CSV)
df_catalog['datetime'] = pd.to_datetime(df_catalog['time_utc'], utc=True)
df_catalog['date'] = df_catalog['datetime'].dt.date
df_catalog['year'] = df_catalog['datetime'].dt.year
df_catalog['event_id'] = df_catalog['event_id'].astype(str)
print(f"✅ Total event di katalog: {len(df_catalog)}")

# Filter katalog
MIN_MAGNITUDE = 4.5
MIN_YEAR = 2004
df_filtered = df_catalog[
    (df_catalog['magnitude'] >= MIN_MAGNITUDE) &
    (df_catalog['year'] >= MIN_YEAR)
].copy()
print(f"✅ Event di katalog setelah filter: {len(df_filtered)}")

# =============================================
# 3. COCOKKAN BERDASARKAN TANGGAL
# =============================================

print(f"\n📊 Mencocokkan event berdasarkan tanggal (toleransi {TIME_TOLERANCE_HOURS} jam)...")

# Buat dictionary untuk akses cepat
catalog_by_date = {}
for idx, row in df_filtered.iterrows():
    date_key = row['date']
    if date_key not in catalog_by_date:
        catalog_by_date[date_key] = []
    catalog_by_date[date_key].append(row)

matched_events = []
unmatched_count = 0
matched_count = 0

for _, file_row in df_files.iterrows():
    file_date = file_row['date']
    file_time = file_row['datetime']
    
    # Cari di katalog dengan tanggal yang sama atau ±1 hari
    found = False
    for offset in [0, -1, 1]:
        check_date = file_date + timedelta(days=offset)
        if check_date in catalog_by_date:
            for cat_row in catalog_by_date[check_date]:
                time_diff = abs((cat_row['datetime'] - file_time).total_seconds()) / 3600  # jam
                if time_diff <= TIME_TOLERANCE_HOURS:
                    matched_events.append({
                        'file': file_row['file'],
                        'network': file_row['network'],
                        'station': file_row['station'],
                        'file_time': file_row['datetime'],
                        'catalog_time': cat_row['datetime'],
                        'time_diff_hours': time_diff,
                        'event_id': cat_row['event_id'],
                        'magnitude': cat_row['magnitude'],
                        'depth_km': cat_row['depth_km'],
                        'latitude': cat_row['latitude'],
                        'longitude': cat_row['longitude'],
                        'source': cat_row['source']
                    })
                    matched_count += 1
                    found = True
                    break
        if found:
            break
    
    if not found:
        unmatched_count += 1

df_matched = pd.DataFrame(matched_events)
print(f"✅ Event tercocokkan: {matched_count}")
print(f"❌ Event tidak tercocokkan: {unmatched_count}")
print(f"📊 Success rate: {matched_count/(matched_count+unmatched_count)*100:.1f}%")

# =============================================
# 4. STATISTIK
# =============================================

print("\n" + "="*70)
print("📊 STATISTIK KECOCOKAN")
print("="*70)

if matched_count == 0:
    print("⚠️ Tidak ada event yang tercocokkan!")
    print("Kemungkinan: tanggal di file tidak match dengan katalog.")
    exit()

# --- Distribusi stasiun ---
print("\n📡 10 Stasiun Terbanyak:")
for sta, count in df_matched['station'].value_counts().head(10).items():
    print(f"  {sta}: {count} ({count/len(df_matched)*100:.1f}%)")

# --- Distribusi magnitude ---
print(f"\n📊 Magnitudo rata-rata: {df_matched['magnitude'].mean():.2f}")
print(f"   Min: {df_matched['magnitude'].min():.1f}, Max: {df_matched['magnitude'].max():.1f}")

# --- Distribusi tahun ---
print(f"\n📊 Distribusi tahun:")
yearly = df_matched['year'].value_counts().sort_index()
for year, count in yearly.items():
    print(f"  {year}: {count}")

# =============================================
# 5. VISUALISASI
# =============================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 5a. Timeline tahunan
ax1 = axes[0, 0]
ax1.bar(yearly.index, yearly.values, color='steelblue', alpha=0.7)
ax1.set_xlabel('Tahun')
ax1.set_ylabel('Jumlah Event')
ax1.set_title('Distribusi Event per Tahun')
ax1.grid(True, alpha=0.3)

# 5b. Distribusi magnitudo
ax2 = axes[0, 1]
ax2.hist(df_matched['magnitude'], bins=20, color='coral', alpha=0.7, edgecolor='black')
ax2.set_xlabel('Magnitudo')
ax2.set_ylabel('Frekuensi')
ax2.set_title('Distribusi Magnitudo')
ax2.grid(True, alpha=0.3)

# 5c. Top stations
ax3 = axes[1, 0]
top_stations = df_matched['station'].value_counts().head(15)
ax3.barh(top_stations.index, top_stations.values, color='green', alpha=0.7)
ax3.set_xlabel('Jumlah File')
ax3.set_title('15 Stasiun Terbanyak')
ax3.grid(True, alpha=0.3)

# 5d. Pie chart jaringan
ax4 = axes[1, 1]
network_counts = df_matched['network'].value_counts()
ax4.pie(network_counts.values, labels=network_counts.index, autopct='%1.1f%%')
ax4.set_title('Distribusi Jaringan')

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/matched_analysis.png", dpi=300, bbox_inches='tight')
plt.close()
print(f"📊 Matched analysis: {OUTPUT_DIR}/matched_analysis.png")

# =============================================
# 6. SIMPAN HASIL
# =============================================

df_matched.to_csv(f"{OUTPUT_DIR}/matched_events.csv", index=False)
print(f"✅ Daftar event tercocokkan: {OUTPUT_DIR}/matched_events.csv")

# =============================================
# 7. RINGKASAN
# =============================================

print("\n" + "="*70)
print("📊 RINGKASAN VERIFIKASI")
print("="*70)
print(f"Total file terunduh: {len(files)}")
print(f"Event unik dari file: {len(df_files)}")
print(f"Event tercocokkan: {matched_count} ({matched_count/len(df_files)*100:.1f}%)")
print(f"Event tidak tercocokkan: {unmatched_count} ({unmatched_count/len(df_files)*100:.1f}%)")
print("="*70)

🔍 VERIFIKASI KECOCOKAN (BERDASARKAN TANGGAL + 12 JAM)

📂 Membaca file .mseed...
✅ Total file .mseed valid: 20215
✅ Event unik terunduh: 20215

📂 Membaca katalog...
✅ Total event di katalog: 232301
✅ Event di katalog setelah filter: 25438

📊 Mencocokkan event berdasarkan tanggal (toleransi 12 jam)...
✅ Event tercocokkan: 16454
❌ Event tidak tercocokkan: 3761
📊 Success rate: 81.4%

📊 STATISTIK KECOCOKAN

📡 10 Stasiun Terbanyak:
  UGM: 3434 (20.9%)
  TNTI: 2307 (14.0%)
  GSI: 1634 (9.9%)
  PMG: 1612 (9.8%)
  MNAI: 995 (6.0%)
  BNDI: 828 (5.0%)
  MMRI: 790 (4.8%)
  LUWI: 649 (3.9%)
  FAKI: 617 (3.7%)
  CISI: 499 (3.0%)

📊 Magnitudo rata-rata: 4.90
   Min: 4.5, Max: 9.1

📊 Distribusi tahun:


KeyError: 'year'

In [13]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
VERIFIKASI KECOCOKAN FILE TERUNDUH DENGAN KATALOG (FIXED)
Menggunakan waktu dan koordinat untuk mencocokkan.
"""

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import timedelta
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# =============================================
# 1. KONFIGURASI
# =============================================

WAVEFORM_DIR = "/Volumes/Extreme SSD/unduhan_waveform_geofon_juli"
CATALOG_CSV = "/Volumes/Extreme SSD/unduhan_juni_bmkg_usgs/hasilscan/eda_final_output/HYBRID_EARTHQUAKE_CATALOG_2001_2024_FIX.csv"
OUTPUT_DIR = "verifikasi_katalog_fixed"
Path(OUTPUT_DIR).mkdir(exist_ok=True, parents=True)

TIME_TOLERANCE_HOURS = 12  # toleransi 12 jam

# =============================================
# 2. BACA DATA
# =============================================

print("="*70)
print("🔍 VERIFIKASI KECOCOKAN (BERDASARKAN TANGGAL + 12 JAM)")
print("="*70)

# --- 2a. Baca file .mseed ---
print("\n📂 Membaca file .mseed...")
files = list(Path(WAVEFORM_DIR).glob("*.mseed"))
files = [f for f in files if not f.name.startswith('._')]
print(f"✅ Total file .mseed valid: {len(files)}")

# Ekstrak waktu dari nama file (format YYYYMMDD)
downloaded_events = []
for f in files:
    parts = f.stem.split('_')
    if len(parts) >= 3:
        for p in parts:
            if len(p) == 14 and p.isdigit():
                try:
                    dt = pd.to_datetime(p, format='%Y%m%d%H%M%S', utc=True)
                    downloaded_events.append({
                        'file': f.name,
                        'network': parts[0],
                        'station': parts[1],
                        'datetime': dt,
                        'date': dt.date(),
                        'year': dt.year
                    })
                    break
                except:
                    continue
            elif len(p) == 8 and p.isdigit():
                try:
                    dt = pd.to_datetime(p, format='%Y%m%d', utc=True)
                    downloaded_events.append({
                        'file': f.name,
                        'network': parts[0],
                        'station': parts[1],
                        'datetime': dt,
                        'date': dt.date(),
                        'year': dt.year
                    })
                    break
                except:
                    continue

df_files = pd.DataFrame(downloaded_events)
print(f"✅ Event unik terunduh: {len(df_files)}")

if len(df_files) == 0:
    print("❌ Tidak ada event yang bisa diekstrak dari nama file!")
    exit()

# --- 2b. Baca katalog ---
print("\n📂 Membaca katalog...")
df_catalog = pd.read_csv(CATALOG_CSV)
df_catalog['datetime'] = pd.to_datetime(df_catalog['time_utc'], utc=True)
df_catalog['date'] = df_catalog['datetime'].dt.date
df_catalog['year'] = df_catalog['datetime'].dt.year
print(f"✅ Total event di katalog: {len(df_catalog)}")

# Filter katalog sesuai parameter unduhan
MIN_MAGNITUDE = 4.5
MIN_YEAR = 2004
df_filtered = df_catalog[
    (df_catalog['magnitude'] >= MIN_MAGNITUDE) &
    (df_catalog['year'] >= MIN_YEAR)
].copy()
print(f"✅ Event di katalog setelah filter: {len(df_filtered)}")

# =============================================
# 3. COCOKKAN BERDASARKAN TANGGAL (toleransi 12 jam)
# =============================================

print(f"\n📊 Mencocokkan event berdasarkan tanggal (toleransi {TIME_TOLERANCE_HOURS} jam)...")

# Buat dictionary untuk akses cepat berdasarkan tanggal
catalog_by_date = {}
for idx, row in df_filtered.iterrows():
    date_key = row['date']
    if date_key not in catalog_by_date:
        catalog_by_date[date_key] = []
    catalog_by_date[date_key].append(row)

# Buat dictionary untuk akses cepat berdasarkan event_id (untuk fallback)
catalog_by_event = {row['event_id']: row for idx, row in df_filtered.iterrows()}

# Cari kecocokan
matched_events = []
unmatched_events = []

for _, file_row in df_files.iterrows():
    file_date = file_row['date']
    file_time = file_row['datetime']
    
    best_match = None
    best_diff = float('inf')
    
    # Cari di tanggal yang sama
    if file_date in catalog_by_date:
        for cat_row in catalog_by_date[file_date]:
            time_diff = abs((cat_row['datetime'] - file_time).total_seconds())
            if time_diff < best_diff:
                best_diff = time_diff
                best_match = cat_row
    
    # Jika tidak ada di tanggal yang sama, cari di tanggal ±1 hari
    if best_match is None:
        for delta in [-1, 1]:
            check_date = file_date + timedelta(days=delta)
            if check_date in catalog_by_date:
                for cat_row in catalog_by_date[check_date]:
                    time_diff = abs((cat_row['datetime'] - file_time).total_seconds())
                    if time_diff < best_diff:
                        best_diff = time_diff
                        best_match = cat_row
    
    # Jika tetap tidak ada, coba gunakan event_id dari nama file
    if best_match is None:
        # Coba ekstrak event_id dari nama file (format lain)
        for p in file_row['file'].split('_'):
            if p.startswith('usp') or p.startswith('official') or p.startswith('BMKG'):
                if p in catalog_by_event:
                    best_match = catalog_by_event[p]
                    break
    
    if best_match is not None:
        matched_events.append({
            'file': file_row['file'],
            'network': file_row['network'],
            'station': file_row['station'],
            'file_time': file_row['datetime'],
            'catalog_time': best_match['datetime'],
            'time_diff': best_diff,
            'event_id': best_match['event_id'],
            'magnitude': best_match['magnitude'],
            'depth_km': best_match['depth_km'],
            'latitude': best_match['latitude'],
            'longitude': best_match['longitude'],
            'source': best_match['source'],
            'year': best_match['year'],  # Tambahkan kolom year
            'matched': True
        })
    else:
        unmatched_events.append({
            'file': file_row['file'],
            'network': file_row['network'],
            'station': file_row['station'],
            'file_time': file_row['datetime'],
            'year': file_row['year'],
            'matched': False
        })

df_matched = pd.DataFrame(matched_events)
df_unmatched = pd.DataFrame(unmatched_events)

print(f"✅ Event tercocokkan: {len(df_matched)}")
print(f"❌ Event tidak tercocokkan: {len(df_unmatched)}")
print(f"📊 Success rate: {len(df_matched)/len(df_files)*100:.1f}%")

# =============================================
# 4. STATISTIK
# =============================================

print("\n" + "="*70)
print("📊 STATISTIK KECOCOKAN")
print("="*70)

if len(df_matched) == 0:
    print("⚠️ Tidak ada event yang tercocokkan!")
    exit()

# --- Distribusi stasiun ---
print("\n📡 10 Stasiun Terbanyak:")
for sta, count in df_matched['station'].value_counts().head(10).items():
    print(f"  {sta}: {count} ({count/len(df_matched)*100:.1f}%)")

# --- Distribusi magnitude ---
print(f"\n📊 Magnitudo rata-rata: {df_matched['magnitude'].mean():.2f}")
print(f"   Min: {df_matched['magnitude'].min():.1f}, Max: {df_matched['magnitude'].max():.1f}")

# --- Distribusi tahun ---
print(f"\n📊 Distribusi tahun:")
yearly = df_matched['year'].value_counts().sort_index()
for year, count in yearly.items():
    print(f"  {year}: {count}")

# --- Distribusi sumber ---
print(f"\n📊 Distribusi sumber:")
for source, count in df_matched['source'].value_counts().items():
    print(f"  {source}: {count} ({count/len(df_matched)*100:.1f}%)")

# =============================================
# 5. VISUALISASI
# =============================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 5a. Timeline tahunan
ax1 = axes[0, 0]
ax1.bar(yearly.index, yearly.values, color='steelblue', alpha=0.7)
ax1.set_xlabel('Tahun')
ax1.set_ylabel('Jumlah Event')
ax1.set_title('Distribusi Event per Tahun')
ax1.grid(True, alpha=0.3)

# 5b. Distribusi magnitudo
ax2 = axes[0, 1]
ax2.hist(df_matched['magnitude'], bins=20, color='coral', alpha=0.7, edgecolor='black')
ax2.set_xlabel('Magnitudo')
ax2.set_ylabel('Frekuensi')
ax2.set_title('Distribusi Magnitudo')
ax2.grid(True, alpha=0.3)

# 5c. Top stations
ax3 = axes[1, 0]
top_stations = df_matched['station'].value_counts().head(15)
ax3.barh(top_stations.index, top_stations.values, color='green', alpha=0.7)
ax3.set_xlabel('Jumlah File')
ax3.set_title('15 Stasiun Terbanyak')
ax3.grid(True, alpha=0.3)

# 5d. Pie chart sumber
ax4 = axes[1, 1]
source_counts = df_matched['source'].value_counts()
ax4.pie(source_counts.values, labels=source_counts.index, autopct='%1.1f%%')
ax4.set_title('Distribusi Sumber Data')

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/matched_analysis.png", dpi=300, bbox_inches='tight')
plt.close()
print(f"📊 Matched analysis: {OUTPUT_DIR}/matched_analysis.png")

# =============================================
# 6. SIMPAN HASIL
# =============================================

df_matched.to_csv(f"{OUTPUT_DIR}/matched_events.csv", index=False)
df_unmatched.to_csv(f"{OUTPUT_DIR}/unmatched_events.csv", index=False)
print(f"✅ Daftar event tercocokkan: {OUTPUT_DIR}/matched_events.csv")
print(f"✅ Daftar event tidak tercocokkan: {OUTPUT_DIR}/unmatched_events.csv")

# =============================================
# 7. RINGKASAN
# =============================================

print("\n" + "="*70)
print("📊 RINGKASAN VERIFIKASI")
print("="*70)
print(f"Total file terunduh: {len(files)}")
print(f"Event unik dari nama file: {len(df_files)}")
print(f"Event tercocokkan dengan katalog: {len(df_matched)} ({len(df_matched)/len(df_files)*100:.1f}%)")
print(f"Event tidak tercocokkan: {len(df_unmatched)} ({len(df_unmatched)/len(df_files)*100:.1f}%)")
print("="*70)

🔍 VERIFIKASI KECOCOKAN (BERDASARKAN TANGGAL + 12 JAM)

📂 Membaca file .mseed...
✅ Total file .mseed valid: 20215
✅ Event unik terunduh: 20215

📂 Membaca katalog...
✅ Total event di katalog: 232301
✅ Event di katalog setelah filter: 25438

📊 Mencocokkan event berdasarkan tanggal (toleransi 12 jam)...
✅ Event tercocokkan: 17206
❌ Event tidak tercocokkan: 3009
📊 Success rate: 85.1%

📊 STATISTIK KECOCOKAN

📡 10 Stasiun Terbanyak:
  UGM: 3522 (20.5%)
  TNTI: 2416 (14.0%)
  GSI: 1709 (9.9%)
  PMG: 1690 (9.8%)
  MNAI: 1043 (6.1%)
  BNDI: 889 (5.2%)
  MMRI: 839 (4.9%)
  LUWI: 686 (4.0%)
  FAKI: 636 (3.7%)
  CISI: 534 (3.1%)

📊 Magnitudo rata-rata: 4.90
   Min: 4.5, Max: 9.1

📊 Distribusi tahun:
  2004: 2024
  2005: 2712
  2006: 2479
  2007: 2058
  2008: 1760
  2009: 3852
  2010: 2321

📊 Distribusi sumber:
  USGS: 11033 (64.1%)
  BMKG: 6173 (35.9%)
📊 Matched analysis: verifikasi_katalog_fixed/matched_analysis.png
✅ Daftar event tercocokkan: verifikasi_katalog_fixed/matched_events.csv
✅ Daftar e

In [14]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
INVESTIGASI KOMPONEN FILE .mseed
Mengecek apakah file memiliki komponen Z, N, E (3 komponen)
"""

import os
import numpy as np
import pandas as pd
from pathlib import Path
from obspy import read
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# =============================================
# 1. KONFIGURASI
# =============================================

WAVEFORM_DIR = "/Volumes/Extreme SSD/unduhan_waveform_geofon_juli"
SAMPLE_SIZE = 500  # jumlah file yang diambil untuk sampling
OUTPUT_DIR = "investigasi_komponen"
Path(OUTPUT_DIR).mkdir(exist_ok=True, parents=True)

# =============================================
# 2. BACA FILE SAMPLE
# =============================================

print("="*70)
print("🔍 INVESTIGASI KOMPONEN FILE .mseed")
print("="*70)

# Cari semua file .mseed
files = list(Path(WAVEFORM_DIR).glob("*.mseed"))
files = [f for f in files if not f.name.startswith('._')]
print(f"✅ Total file .mseed valid: {len(files)}")

# Ambil sample acak
if len(files) > SAMPLE_SIZE:
    import random
    sample_files = random.sample(files, SAMPLE_SIZE)
else:
    sample_files = files

print(f"📊 Mengambil sample {len(sample_files)} file untuk investigasi")

# =============================================
# 3. EKSTRAK KOMPONEN
# =============================================

results = []
components_summary = []

for f in sample_files:
    try:
        st = read(str(f))
        if len(st) == 0:
            continue
        
        # Kumpulkan channel yang tersedia
        channels = []
        components = {'Z': False, 'N': False, 'E': False}
        comp_list = []
        
        for tr in st:
            ch = tr.stats.channel
            channels.append(ch)
            # Cek komponen
            if ch.endswith('Z'):
                components['Z'] = True
                comp_list.append('Z')
            elif ch.endswith('N'):
                components['N'] = True
                comp_list.append('N')
            elif ch.endswith('E'):
                components['E'] = True
                comp_list.append('E')
        
        # Tentukan status
        has_z = components['Z']
        has_n = components['N']
        has_e = components['E']
        
        # Kategori
        if has_z and has_n and has_e:
            category = "3C (Z,N,E)"
        elif has_z and (has_n or has_e):
            category = "2C (Z + N/E)"
        elif has_z:
            category = "1C (Z only)"
        else:
            category = "No Z component"
        
        results.append({
            'file': f.name,
            'network': st[0].stats.network if len(st) > 0 else 'UNK',
            'station': st[0].stats.station if len(st) > 0 else 'UNK',
            'channels': ','.join(channels),
            'has_Z': has_z,
            'has_N': has_n,
            'has_E': has_e,
            'n_traces': len(st),
            'category': category,
            'comp_list': ','.join(comp_list) if comp_list else 'none'
        })
        
        components_summary.append(comp_list)
        
    except Exception as e:
        continue

df = pd.DataFrame(results)
print(f"✅ Berhasil diproses: {len(df)} file")

# =============================================
# 4. STATISTIK
# =============================================

print("\n" + "="*70)
print("📊 STATISTIK KOMPONEN")
print("="*70)

# Distribusi kategori
print("\n📊 Kategori komponen:")
for cat, count in df['category'].value_counts().items():
    print(f"  {cat}: {count} ({count/len(df)*100:.1f}%)")

# File dengan 3 komponen
df_3c = df[df['category'] == "3C (Z,N,E)"]
print(f"\n✅ File dengan 3 komponen (Z,N,E): {len(df_3c)} ({len(df_3c)/len(df)*100:.1f}%)")

# File dengan komponen Z
df_has_z = df[df['has_Z']]
print(f"✅ File dengan komponen Z: {len(df_has_z)} ({len(df_has_z)/len(df)*100:.1f}%)")

# File tanpa Z
df_no_z = df[~df['has_Z']]
if len(df_no_z) > 0:
    print(f"⚠️ File tanpa komponen Z: {len(df_no_z)} ({len(df_no_z)/len(df)*100:.1f}%)")
    print("   Contoh file tanpa Z:")
    for _, row in df_no_z.head(5).iterrows():
        print(f"     - {row['file']} (channels: {row['channels']})")

# =============================================
# 5. VISUALISASI
# =============================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 5a. Pie chart kategori
ax1 = axes[0, 0]
cat_counts = df['category'].value_counts()
colors_pie = ['#2ecc71' if '3C' in c else '#f39c12' if '2C' in c else '#e74c3c' if 'No Z' in c else '#3498db' 
              for c in cat_counts.index]
ax1.pie(cat_counts.values, labels=cat_counts.index, autopct='%1.1f%%', colors=colors_pie, startangle=90)
ax1.set_title('Kategori Komponen', fontsize=14, fontweight='bold')

# 5b. Bar chart: kepemilikan komponen
ax2 = axes[0, 1]
has_z = df['has_Z'].sum()
has_n = df['has_N'].sum()
has_e = df['has_E'].sum()
total = len(df)

components = ['Z', 'N', 'E']
counts = [has_z, has_n, has_e]
colors_bar = ['#2ecc71', '#3498db', '#e74c3c']
bars = ax2.bar(components, counts, color=colors_bar, alpha=0.7)
ax2.set_xlabel('Komponen')
ax2.set_ylabel('Jumlah File')
ax2.set_title('Ketersediaan Komponen per File', fontsize=14, fontweight='bold')
for bar, val in zip(bars, counts):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 10, f'{val} ({val/total*100:.1f}%)', 
             ha='center', va='bottom', fontsize=9)
ax2.grid(True, alpha=0.3, axis='y')

# 5c. Kombinasi komponen yang paling sering muncul
ax3 = axes[1, 0]
comp_combos = df['comp_list'].value_counts().head(10)
ax3.barh(comp_combos.index, comp_combos.values, color='steelblue', alpha=0.7)
ax3.set_xlabel('Jumlah File')
ax3.set_title('Kombinasi Komponen Terbanyak', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3, axis='x')
for i, (combo, count) in enumerate(comp_combos.items()):
    ax3.text(count + 5, i, f'{count} ({count/len(df)*100:.1f}%)', va='center', fontsize=8)

# 5d. Boxplot jumlah trace per file
ax4 = axes[1, 1]
ax4.boxplot(df['n_traces'])
ax4.set_xlabel('File .mseed')
ax4.set_ylabel('Jumlah Trace')
ax4.set_title('Distribusi Jumlah Trace per File', fontsize=14, fontweight='bold')
ax4.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/components_analysis.png", dpi=300, bbox_inches='tight')
plt.close()
print(f"\n📊 Components analysis: {OUTPUT_DIR}/components_analysis.png")

# =============================================
# 6. SIMPAN HASIL
# =============================================

df.to_csv(f"{OUTPUT_DIR}/components_report.csv", index=False)
print(f"✅ Components report: {OUTPUT_DIR}/components_report.csv")

# =============================================
# 7. RINGKASAN
# =============================================

print("\n" + "="*70)
print("📊 RINGKASAN INVESTIGASI KOMPONEN")
print("="*70)
print(f"Total file sample: {len(df)}")
print(f"File dengan 3 komponen (Z,N,E): {len(df_3c)} ({len(df_3c)/len(df)*100:.1f}%)")
print(f"File dengan komponen Z: {len(df_has_z)} ({len(df_has_z)/len(df)*100:.1f}%)")
print(f"File tanpa komponen Z: {len(df_no_z)} ({len(df_no_z)/len(df)*100:.1f}%)")
print(f"Rata-rata jumlah trace per file: {df['n_traces'].mean():.1f}")
print("="*70)

🔍 INVESTIGASI KOMPONEN FILE .mseed
✅ Total file .mseed valid: 20215
📊 Mengambil sample 500 file untuk investigasi
✅ Berhasil diproses: 500 file

📊 STATISTIK KOMPONEN

📊 Kategori komponen:
  3C (Z,N,E): 494 (98.8%)
  1C (Z only): 4 (0.8%)
  No Z component: 2 (0.4%)

✅ File dengan 3 komponen (Z,N,E): 494 (98.8%)
✅ File dengan komponen Z: 498 (99.6%)
⚠️ File tanpa komponen Z: 2 (0.4%)
   Contoh file tanpa Z:
     - GE_MMRI_20080129_042006.mseed (channels: HHN,HHE)
     - GE_GSI_20060426_031327.mseed (channels: BHN,BHE)

📊 Components analysis: investigasi_komponen/components_analysis.png
✅ Components report: investigasi_komponen/components_report.csv

📊 RINGKASAN INVESTIGASI KOMPONEN
Total file sample: 500
File dengan 3 komponen (Z,N,E): 494 (98.8%)
File dengan komponen Z: 498 (99.6%)
File tanpa komponen Z: 2 (0.4%)
Rata-rata jumlah trace per file: 4.1


In [18]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
MENGABUNGKAN DATASET UTAMA DAN GEMPA MERUSAK - FINAL
"""

import os
import re
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# =============================================
# 1. KONFIGURASI
# =============================================

DIR_MAIN = "/Volumes/Extreme SSD/unduhan_waveform_geofon_juli"
DIR_MAJOR = "/Volumes/Extreme SSD/unduhan_waveform_major_earthquakes"
OUTPUT_CSV = "/Volumes/Extreme SSD/unduhan_waveform_merged/file_list_merged_final.csv"
OUTPUT_DIR = "/Volumes/Extreme SSD/unduhan_waveform_merged"

Path(OUTPUT_DIR).mkdir(exist_ok=True, parents=True)

def extract_timestamp(filename):
    """Ekstrak timestamp (YYYYMMDD) dari berbagai format nama file."""
    # Pola 1: BMKG-YYYYMMDD...
    match = re.search(r'BMKG-(\d{8})', filename)
    if match:
        return match.group(1)
    # Pola 2: officialYYYYMMDD...
    match = re.search(r'official(\d{8})', filename)
    if match:
        return match.group(1)
    # Pola 3: 8 digit angka di mana pun
    match = re.search(r'(\d{8})', filename)
    if match:
        return match.group(1)
    return None

print("="*70)
print("📊 MENGABUNGKAN DATASET - FINAL")
print("="*70)

# --- 1. Baca file ---
files_main = [f for f in Path(DIR_MAIN).glob("*.mseed") if not f.name.startswith('._')]
files_major = [f for f in Path(DIR_MAJOR).glob("*.mseed") if not f.name.startswith('._')]

print(f"\n✅ Batch utama: {len(files_main)} file")
print(f"✅ Batch gempa merusak: {len(files_major)} file")

# --- 2. Gabungkan dan ekstrak metadata ---
all_files = files_main + files_major
metadata = []

for f in all_files:
    parts = f.stem.split('_')
    if len(parts) >= 2:
        network = parts[0]
        station = parts[1]
    else:
        network = 'UNK'
        station = 'UNK'
    
    timestamp = extract_timestamp(f.name)
    source = 'main' if DIR_MAIN in str(f.parent) else 'major'
    
    metadata.append({
        'file': f.name,
        'path': str(f),
        'network': network,
        'station': station,
        'timestamp': timestamp,
        'source': source
    })

df = pd.DataFrame(metadata)
print(f"\n✅ Total metadata diekstrak: {len(df)} file")

# --- 3. Statistik ---
print("\n" + "="*70)
print("📊 STATISTIK DATASET GABUNGAN")
print("="*70)

print(f"Total file: {len(df)}")
print(f"  - Batch utama: {len(df[df['source']=='main'])}")
print(f"  - Gempa merusak: {len(df[df['source']=='major'])}")

# File dengan timestamp tidak valid
invalid_timestamp = df[df['timestamp'].isna()]
if len(invalid_timestamp) > 0:
    print(f"\n⚠️ File dengan timestamp tidak valid: {len(invalid_timestamp)}")
    print("  Contoh:")
    for _, row in invalid_timestamp.head(10).iterrows():
        print(f"    - {row['file']}")

# Distribusi stasiun
print("\n📡 10 Stasiun Terbanyak:")
for sta, count in df['station'].value_counts().head(10).items():
    print(f"  {sta}: {count} ({count/len(df)*100:.1f}%)")

# Distribusi tahun
df_valid = df[df['timestamp'].notna()]
df_valid['year'] = df_valid['timestamp'].str[:4]
yearly = df_valid['year'].value_counts().sort_index()
print("\n📊 Distribusi per Tahun (file dengan timestamp valid):")
for year, count in yearly.items():
    print(f"  {year}: {count}")

if len(invalid_timestamp) > 0:
    print(f"\n⚠️ {len(invalid_timestamp)} file tidak memiliki timestamp yang valid")
    print("   File-file ini tetap ada di dataset tetapi tidak masuk statistik tahun.")

# --- 4. Simpan ---
df.to_csv(OUTPUT_CSV, index=False)
print(f"\n✅ Daftar file gabungan: {OUTPUT_CSV}")

# --- 5. Visualisasi ---
if len(yearly) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    ax1 = axes[0, 0]
    source_counts = df['source'].value_counts()
    ax1.pie(source_counts.values, labels=source_counts.index, autopct='%1.1f%%')
    ax1.set_title('Sumber Data', fontsize=14)
    
    ax2 = axes[0, 1]
    top_stations = df['station'].value_counts().head(10)
    ax2.barh(top_stations.index, top_stations.values, color='steelblue', alpha=0.7)
    ax2.set_xlabel('Jumlah File')
    ax2.set_title('10 Stasiun Terbanyak', fontsize=14)
    
    ax3 = axes[1, 0]
    ax3.bar(yearly.index, yearly.values, color='coral', alpha=0.7)
    ax3.set_xlabel('Tahun')
    ax3.set_ylabel('Jumlah File')
    ax3.set_title('Distribusi per Tahun', fontsize=14)
    
    ax4 = axes[1, 1]
    ax4.axis('tight')
    ax4.axis('off')
    summary = [
        ['Total file', str(len(df))],
        ['Batch utama', str(len(df[df['source']=='main']))],
        ['Gempa merusak', str(len(df[df['source']=='major']))],
        ['Stasiun terbanyak', df['station'].value_counts().index[0]],
        ['Rentang tahun', f"{yearly.index.min()} - {yearly.index.max()}"],
        ['File tanpa timestamp', str(len(invalid_timestamp))],
    ]
    table = ax4.table(cellText=summary, colLabels=['Metrik', 'Nilai'],
                      cellLoc='center', loc='center',
                      colColours=['#4472C4', '#4472C4'])
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 1.8)
    ax4.set_title('Ringkasan Dataset Gabungan', fontsize=14, pad=20)
    
    plt.tight_layout()
    plt.savefig("merged_dataset_final.png", dpi=300, bbox_inches='tight')
    plt.close()
    print("✅ Grafik tersimpan: merged_dataset_final.png")

print("\n" + "="*70)
print("📊 RINGKASAN")
print("="*70)
print(f"Total file unik: {len(df)}")
print(f"File dengan timestamp valid: {len(df_valid)}")
print(f"File tanpa timestamp valid: {len(invalid_timestamp)}")
if len(yearly) > 0:
    print(f"Rentang tahun: {yearly.index.min()} - {yearly.index.max()}")
print(f"Jumlah stasiun unik: {df['station'].nunique()}")
print("="*70)

📊 MENGABUNGKAN DATASET - FINAL

✅ Batch utama: 20215 file
✅ Batch gempa merusak: 17 file

✅ Total metadata diekstrak: 20232 file

📊 STATISTIK DATASET GABUNGAN
Total file: 20232
  - Batch utama: 20215
  - Gempa merusak: 17

⚠️ File dengan timestamp tidak valid: 2
  Contoh:
    - Tsunami_Pangandaran_2006_XN_BUM_usp000ensm.mseed
    - Gempa_Yogyakarta_2006_GE_YOGI_usp000ej1c.mseed

📡 10 Stasiun Terbanyak:
  UGM: 5624 (27.8%)
  PMG: 2586 (12.8%)
  TNTI: 2420 (12.0%)
  GSI: 1709 (8.4%)
  MNAI: 1044 (5.2%)
  BNDI: 892 (4.4%)
  MMRI: 839 (4.1%)
  LUWI: 686 (3.4%)
  FAKI: 636 (3.1%)
  CISI: 535 (2.6%)

📊 Distribusi per Tahun (file dengan timestamp valid):
  2001: 1089
  2002: 1242
  2003: 670
  2004: 2021
  2005: 2713
  2006: 2479
  2007: 2061
  2008: 1764
  2009: 3859
  2010: 2323
  2012: 1
  2016: 1
  2018: 2
  2019: 1
  2021: 1
  2022: 1
  2023: 1
  2024: 1

⚠️ 2 file tidak memiliki timestamp yang valid
   File-file ini tetap ada di dataset tetapi tidak masuk statistik tahun.

✅ Daftar file

In [17]:
import os
from pathlib import Path

DIR_MAJOR = "/Volumes/Extreme SSD/unduhan_waveform_major_earthquakes"

files = list(Path(DIR_MAJOR).glob("*.mseed"))
print(f"Total file di major: {len(files)}")
for f in files:
    print(f.name)

Total file di major: 34
Tsunami_Pangandaran_2006_XN_BUM_usp000ensm.mseed
._Tsunami_Pangandaran_2006_XN_BUM_usp000ensm.mseed
Tsunami_Aceh_2004_GE_UGM_official20041226005853450_30.mseed
._Tsunami_Aceh_2004_GE_UGM_official20041226005853450_30.mseed
Gempa_Bengkulu_2007_GE_MNAI_official20070912111026830_34.mseed
._Gempa_Bengkulu_2007_GE_MNAI_official20070912111026830_34.mseed
Gempa_Padang_2009_GE_BKNI_BMKG-20090930101610-001.mseed
._Gempa_Padang_2009_GE_BKNI_BMKG-20090930101610-001.mseed
Gempa_Sumatra_2010_GE_GSI_BMKG-20100406221503-001.mseed
._Gempa_Sumatra_2010_GE_GSI_BMKG-20100406221503-001.mseed
Gempa_Nias_2005_GE_UGM_official20050328160936530_30.mseed
._Gempa_Nias_2005_GE_UGM_official20050328160936530_30.mseed
Gempa_Yogyakarta_2006_GE_YOGI_usp000ej1c.mseed
._Gempa_Yogyakarta_2006_GE_YOGI_usp000ej1c.mseed
Gempa_Pidie_Jaya_2016_GE_LHMI_BMKG-20161206220334-001.mseed
._Gempa_Pidie_Jaya_2016_GE_LHMI_BMKG-20161206220334-001.mseed
Tsunami_Mentawai_2010_GE_MNAI_BMKG-20101025144221-001.mseed
._

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
INVESTIGASI JSON HASIL EKSTRAKSI
- Cek struktur, label, panjang sinyal, statistik
"""

import json
import numpy as np
from collections import Counter
from pathlib import Path

# =============================================
# 1. KONFIGURASI
# =============================================

JSON_PATH = "/Volumes/Extreme SSD/unduhan_waveform_merged/extracted_data.json"

# =============================================
# 2. LOAD JSON
# =============================================

print("="*70)
print("🔍 INVESTIGASI JSON HASIL EKSTRAKSI")
print("="*70)

with open(JSON_PATH, 'r') as f:
    data = json.load(f)

print(f"\n✅ Total event dalam JSON: {len(data)}")

# =============================================
# 3. CEK STRUKTUR
# =============================================

print("\n📋 CEK STRUKTUR:")

# Ambil satu sample
keys = list(data.keys())
sample_key = keys[0]
sample = data[sample_key]

print(f"  Sample key: {sample_key}")
print(f"  Sample keys: {list(sample.keys())}")
print(f"  Sample type: {sample.get('type', 'missing')}")

# Cek apakah ada 'metadata'
if 'metadata' in sample:
    print(f"  Metadata keys: {list(sample['metadata'].keys())}")

# =============================================
# 4. CEK LABEL
# =============================================

print("\n📊 CEK LABEL:")
label_counts = Counter()
for key, record in data.items():
    label = record.get('type', 'unknown')
    label_counts[label] += 1

for label, count in label_counts.items():
    print(f"  {label}: {count} ({count/len(data)*100:.1f}%)")

# =============================================
# 5. CEK PANJANG SINYAL
# =============================================

print("\n📏 CEK PANJANG SINYAL:")

z_lengths = []
z_noise_lengths = []
no_z_count = 0
no_z_noise_count = 0
corrupt_count = 0

for key, record in data.items():
    z = record.get('Z')
    z_noise = record.get('Z_noise')
    
    if z is None:
        no_z_count += 1
        corrupt_count += 1
        continue
    
    if z_noise is None:
        no_z_noise_count += 1
        corrupt_count += 1
        continue
    
    # Cek panjang
    if len(z) != 700:
        corrupt_count += 1
    z_lengths.append(len(z))
    
    if len(z_noise) != 700:
        corrupt_count += 1
    z_noise_lengths.append(len(z_noise))

print(f"  Jumlah record dengan Z: {len(data) - no_z_count}")
print(f"  Jumlah record dengan Z_noise: {len(data) - no_z_noise_count}")
print(f"  Jumlah record dengan panjang Z != 700: {sum(1 for l in z_lengths if l != 700)}")
print(f"  Jumlah record dengan panjang Z_noise != 700: {sum(1 for l in z_noise_lengths if l != 700)}")
print(f"  Total corrupt: {corrupt_count}")

if z_lengths:
    print(f"  Panjang Z - min: {min(z_lengths)}, max: {max(z_lengths)}, mean: {np.mean(z_lengths):.1f}")

# =============================================
# 6. STATISTIK SINYAL
# =============================================

print("\n📊 STATISTIK SINYAL (sample 100 event):")

sample_keys = list(data.keys())[:100]
z_vals = []
z_noise_vals = []
for key in sample_keys:
    record = data[key]
    z = np.array(record.get('Z', []))
    z_noise = np.array(record.get('Z_noise', []))
    if len(z) == 700:
        z_vals.extend(z)
    if len(z_noise) == 700:
        z_noise_vals.extend(z_noise)

if z_vals:
    print(f"  Z - min: {np.min(z_vals):.4f}, max: {np.max(z_vals):.4f}, mean: {np.mean(z_vals):.4f}, std: {np.std(z_vals):.4f}")
if z_noise_vals:
    print(f"  Z_noise - min: {np.min(z_noise_vals):.4f}, max: {np.max(z_noise_vals):.4f}, mean: {np.mean(z_noise_vals):.4f}, std: {np.std(z_noise_vals):.4f}")

# =============================================
# 7. CEK FORMAT DATA
# =============================================

print("\n📋 CEK FORMAT DATA (sample pertama):")
for key in list(data.keys())[:3]:
    record = data[key]
    z = record.get('Z')
    if z:
        print(f"  {key}: Z length={len(z)}, type={type(z)}, first 3 values={z[:3] if len(z)>=3 else z}")

# =============================================
# 8. RINGKASAN
# =============================================

print("\n" + "="*70)
print("📊 RINGKASAN INVESTIGASI")
print("="*70)
print(f"Total event: {len(data)}")
print(f"Label: {dict(label_counts)}")
print(f"Event dengan Z valid: {len(data) - no_z_count}")
print(f"Event dengan Z_noise valid: {len(data) - no_z_noise_count}")
print(f"Event corrupt (panjang != 700 atau missing): {corrupt_count}")
print(f"Panjang Z - mean: {np.mean(z_lengths):.1f}" if z_lengths else "Panjang Z: N/A")
print("="*70)
print("\n✅ Status JSON: " + ("SEMUA BAIK" if corrupt_count == 0 else f"ADA {corrupt_count} DATA CORRUPT"))

🔍 INVESTIGASI JSON HASIL EKSTRAKSI

✅ Total event dalam JSON: 20164

📋 CEK STRUKTUR:
  Sample key: 20010103_203857_PMG
  Sample keys: ['type', 'Z', 'Z_noise', 'metadata']
  Sample type: se
  Metadata keys: ['network', 'station', 'p_arrival', 'file']

📊 CEK LABEL:
  se: 20164 (100.0%)

📏 CEK PANJANG SINYAL:
  Jumlah record dengan Z: 20164
  Jumlah record dengan Z_noise: 20164
  Jumlah record dengan panjang Z != 700: 0
  Jumlah record dengan panjang Z_noise != 700: 0
  Total corrupt: 0
  Panjang Z - min: 700, max: 700, mean: 700.0

📊 STATISTIK SINYAL (sample 100 event):
  Z - min: nan, max: nan, mean: nan, std: nan
  Z_noise - min: -1.0956, max: 1.2838, mean: 0.2376, std: 0.3111

📋 CEK FORMAT DATA (sample pertama):
  20010103_203857_PMG: Z length=700, type=<class 'list'>, first 3 values=[0.0020621296989864323, 0.003287563272777744, 0.004990716226791131]
  20010102_172154_UGM: Z length=700, type=<class 'list'>, first 3 values=[0.0011134837531645591, 0.0019546947681608863, 0.00312639369331